# Sentiment Analysis using FinViz + VADER
**Financial News Sentiment Analysis with VADER & Trading Recommendations**

This notebook uses FinViz for news scraping and VADER (Valence Aware Dictionary and sEntiment Reasoner) for financial sentiment analysis.

## Key Features:
- Web scraping from FinViz
- VADER sentiment analysis (optimized for financial text)
- Trading signal generation (BUY/SELL/NEUTRAL)
- Multiple ticker comparison
- Sentiment momentum tracking

In [1]:
# !pip install vaderSentiment
import requests
from bs4 import BeautifulSoup
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from typing import List, Dict
from datetime import datetime
import time

print('Libraries imported successfully!')

In [2]:
class FinvizNewsScraper:
    BASE_URL = 'https://finviz.com'
    def __init__(self):
        self.headers = {'User-Agent': 'Mozilla/5.0'}
    def get_news_headlines(self, ticker=None):
        try:
            if ticker:
                url = f'{self.BASE_URL}/quote.ashx?t={ticker.upper()}'
            else:
                url = f'{self.BASE_URL}/news.ashx'
            response = requests.get(url, headers=self.headers, timeout=10)
            soup = BeautifulSoup(response.content, 'lxml')
            news_table = soup.find('table', id='news-table')
            news_items = []
            if news_table:
                rows = news_table.findAll('tr')
                for row in rows:
                    link_cell = row.find('a', class_='tab-link-news')
                    if link_cell:
                        headline = link_cell.getText(strip=True)
                        news_items.append({'headline': headline})
            print(f'Scraped {len(news_items)} headlines')
            return news_items
        except Exception as e:
            print(f'Error: {e}')
            return []

class SentimentAnalyzer:
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()
        # Extended financial lexicon
        financial_lexicon = {
            'beat': 2.0, 'surge': 2.5, 'bullish': 2.5,
            'plunge': -2.5, 'crash': -3.0, 'bearish': -2.5
        }
        self.analyzer.lexicon.update(financial_lexicon)
    def analyze_sentiment(self, text: str):
        return self.analyzer.polarity_scores(text)
    def classify_sentiment(self, compound_score: float):
        if compound_score >= 0.05:
            return 'Positive'
        elif compound_score <= -0.05:
            return 'Negative'
        else:
            return 'Neutral'

class TradingRecommendationEngine:
    THRESHOLDS = {
        'strong_buy': {'avg_sentiment': 0.3, 'positive_ratio': 0.7},
        'buy': {'avg_sentiment': 0.15, 'positive_ratio': 0.6},
        'sell': {'avg_sentiment': -0.15, 'positive_ratio': 0.3},
        'strong_sell': {'avg_sentiment': -0.3, 'positive_ratio': 0.2}
    }
    def calculate_metrics(self, sentiment_scores):
        if not sentiment_scores:
            return {}
        total = len(sentiment_scores)
        positive = sum(1 for s in sentiment_scores if s >= 0.05)
        negative = sum(1 for s in sentiment_scores if s <= -0.05)
        return {
            'avg_sentiment': sum(sentiment_scores) / total,
            'positive_ratio': positive / total,
            'negative_ratio': negative / total,
            'total_headlines': total
        }
    def generate_recommendation(self, metrics):
        avg_sent = metrics['avg_sentiment']
        pos_ratio = metrics['positive_ratio']
        if avg_sent >= self.THRESHOLDS['strong_buy']['avg_sentiment'] and pos_ratio >= self.THRESHOLDS['strong_buy']['positive_ratio']:
            return 'STRONG BUY', f'Positive sentiment {avg_sent:.3f} with {pos_ratio:.1%} positive coverage'
        elif avg_sent >= self.THRESHOLDS['buy']['avg_sentiment'] and pos_ratio >= self.THRESHOLDS['buy']['positive_ratio']:
            return 'BUY', f'Positive sentiment {avg_sent:.3f}'
        elif avg_sent <= self.THRESHOLDS['strong_sell']['avg_sentiment'] and pos_ratio <= self.THRESHOLDS['strong_sell']['positive_ratio']:
            return 'STRONG SELL', f'Negative sentiment {avg_sent:.3f}'
        elif avg_sent <= self.THRESHOLDS['sell']['avg_sentiment']:
            return 'SELL', f'Negative sentiment {avg_sent:.3f}'
        else:
            return 'NEUTRAL', f'Mixed sentiment {avg_sent:.3f}'

print('Classes initialized!')

In [3]:
# Example usage
scraper = FinvizNewsScraper()
sentiment_analyzer = SentimentAnalyzer()
rec_engine = TradingRecommendationEngine()

ticker = 'AAPL'
print(f'Analyzing {ticker}...')
news_items = scraper.get_news_headlines(ticker)

compound_scores = []
for item in news_items[:20]:
    sentiment = sentiment_analyzer.analyze_sentiment(item['headline'])
    item['sentiment'] = sentiment
    item['classification'] = sentiment_analyzer.classify_sentiment(sentiment['compound'])
    compound_scores.append(sentiment['compound'])

metrics = rec_engine.calculate_metrics(compound_scores)
recommendation, rationale = rec_engine.generate_recommendation(metrics)

print(f'\nRESULTS FOR {ticker}:')
print(f'Total Headlines: {metrics["total_headlines"]}')
print(f'Avg Sentiment: {metrics["avg_sentiment"]:.4f}')
print(f'Positive Ratio: {metrics["positive_ratio"]:.1%}')
print(f'RECOMMENDATION: {recommendation}')
print(f'Rationale: {rationale}')